Phys 510 - HW1

Andrew Koren

1. Shift of creationand annihilation operators.

<!-- Check if $b^\dagger b = \bold{1}$, where $b = a + \alpha$. -->



$$
\def\c#1#2{\left[#1,#2\right]}
\def\ac#1#2{\left\{#1,#2\right\}}
\def\dag{\dagger}
      b^\dag
    = a^\dag + \alpha^*
% \\    b^\dag b
%     = \left( a^\dag + \alpha^* \right)
%       \left( a + \alpha \right)
%     = a^\dag a + \alpha^* a
%     + a^\dag \alpha + \alpha \alpha^*
$$

Bosonic case: $[a,a^\dagger]=1$

Fermionic case: $\{a,a^\dagger\}=1$, $a^2 = 1$



For $b$ to remain bosonic, we have

$$
\def\c#1#2{\left[#1,#2\right]}
\def\dag{\dagger}
      \c{b}{b^\dag} 
    = bb^\dag - b^\dag b 
\\  = (a + \alpha)(a^\dag + \alpha^*) - (a^\dag + \alpha^*)(a + \alpha)
$$

It might be easier to use sympy for this

In [1]:
import sympy as sp
from sympy.physics.quantum.dagger import Dagger
from sympy.physics.quantum.operator import Operator

b_symb, α = sp.symbols('b alpha', complex=True)

a = Operator('a')

b = a + α
b_commutator = sp.expand(b*Dagger(b)-Dagger(b)*b)
b_anticommutator = sp.expand(b*Dagger(b)+Dagger(b)*b)

display('commutator:')
display(b_commutator)
display('anticommutator')
display(b_anticommutator)
display(sp.Eq(b_symb*b_symb,sp.expand(b*b)))

'commutator:'

-Dagger(a)*a + a*Dagger(a)

'anticommutator'

2*alpha*conjugate(alpha) + 2*alpha*Dagger(a) + 2*conjugate(alpha)*a + Dagger(a)*a + a*Dagger(a)

Eq(b**2, alpha**2 + 2*alpha*a + a**2)

This gives

$$
\c{b}{b^\dag} = \c{a}{a^\dag} = 1
$$

so the commutation relation is preserved and $b$ can be made via unitary transform. As for the fermionic case, it is clear that the result is only valid for $\alpha = 0$.

<!-- If the tranformation is unitary, then there must exist

$$
    U b U^\dag = a
\\  U^\dag b^\dag U = a^\dag
\\  U^\dag a U = b
\\  U a^\dag U^\dag = b^\dag
$$
yielding
$$
\begin{align*}
        U^\dag ( a +   \alpha) U        &= a 
\\      U^\dag a U +   \alpha U^\dag U  &= a
\\      U^\dag a U +   \alpha           &= a
\\    U U^\dag a U + U \alpha           &= U a 
\\             a U                      &= U (a - \alpha)
\end{align*}
$$
Note that the $\bold{1}$ is omitted


<!-- yielding
$$
\\  b = U^\dag a U
\\  b^\dag = U a^\dag U^\dag
\\  U^\dag b^\dag U = a^\dag
$$

$$
    U ( a + \alpha ) U^\dag 
\\  = U a U^\dag + \alpha U U^\dag
\\  = U a U^\dag + \alpha 
    = a
$$
Note that $\alpha = b-a$, yielding

$$
      U a U^\dag + b - a
    = a
\\    U a U^\dag + b
    = 2a
\\    U a U^\dag + b
    = 2a


    %   U^\dag a^\dag U + \alpha^*
    % = a^\dag
$$ -->

2. Bogolubov

Using real $u$, $v$
$$
      \mathcal{H} 
    = \epsilon \left( a_1^\dag a_1 + a_2^\dag a_2 \right)
    + \lambda  \left( a_1^\dag a_2^\dag + a_2 a_1 \right)
\\  = \epsilon \left( 
          ( u b_1^\dag + v b_2 )( u b_1 + v b_2^\dag )
        + ( u b_2^\dag + v b_1 )( u b_2 + v b_1^\dag )
        \right)
    + \lambda  \left( 
          ( u b_1^\dag + v b_2  ) (u b_2^\dag + v b_1)
        + ( u b_2 + v b_1^\dag  ) (u b_1 + v b_2^\dag)
        \right)
$$

It might be easier to use sympy for this

In [2]:
Hsymbol, ε, λ, u, v = sp.symbols('H epsilon lambda u v', real=True)
b1, b2 = Operator('b_1'), Operator('b_2')

a1d = u * Dagger(b1) + v * b2
a2d = u * Dagger(b2) + v * b1
a1 = Dagger(a1d)
a2 = Dagger(a2d)

#display(a1, a2)
H = ε * (a1d*a1 + a2d*a2) + λ * (a1d*a2d+a2*a1) 
#display(H)
display(sp.Eq(Hsymbol,sp.simplify(sp.expand(H))))

Eq(H, epsilon*u**2*Dagger(b_1)*b_1 + epsilon*u**2*Dagger(b_2)*b_2 + epsilon*u*v*Dagger(b_1)*Dagger(b_2) + epsilon*u*v*Dagger(b_2)*Dagger(b_1) + epsilon*u*v*b_1*b_2 + epsilon*u*v*b_2*b_1 + epsilon*v**2*b_1*Dagger(b_1) + epsilon*v**2*b_2*Dagger(b_2) + lambda*u**2*Dagger(b_1)*Dagger(b_2) + lambda*u**2*b_2*b_1 + 2*lambda*u*v*Dagger(b_1)*b_1 + 2*lambda*u*v*b_2*Dagger(b_2) + lambda*v**2*Dagger(b_1)*Dagger(b_2) + lambda*v**2*b_2*b_1)

I wrote a script to collect all the like terms based on commutation relations. It probably would have been faster to do it by hand, but at least I won't make any mistakes

In [3]:
from operator_module import collect_operators
Hexp = sp.expand(H)

commuting_pairs = {
    frozenset([b1, b2]),
    frozenset([Dagger(b1), Dagger(b2)]),
    frozenset([b1, Dagger(b2)]),
    frozenset([b2, Dagger(b1)]),
}

collected = collect_operators(Hexp, collect_coeffs = True, commuting_pairs=commuting_pairs)

H_collected = 0
for op, coeff in collected.items():
    if coeff != 0:
        H_collected += coeff * op
display(sp.Eq(Hsymbol,H_collected))

Eq(H, epsilon*u**2*Dagger(b_2)*b_2 + epsilon*v**2*b_1*Dagger(b_1) + u*(epsilon*u + 2*lambda*v)*Dagger(b_1)*b_1 + v*(epsilon*v + 2*lambda*u)*b_2*Dagger(b_2) + (2*epsilon*u*v + lambda*u**2 + lambda*v**2)*(Dagger(b_1)*Dagger(b_2) + b_1*b_2))

How do we diagonalize? Just make the off-diagonal coefficients zero.

$$
\begin{align*}
      \hat H 
  & = \underbrace{
     (\epsilon u^2 + 2\lambda v) b_1^\dag b_1 
    + \epsilon v^2 b_1 b_1^\dag
    +(\epsilon v^2 + 2\lambda u) b_2 b_2^\dag 
    + \epsilon u^2 b_2^\dag b_2
    }_{\text{diagonal}}
\\& + \underbrace{
     (2\epsilon uv + \lambda (u^2 + v^2)) \left( b_1^\dag b_2^\dag + b_1 b_2 \right)
    }_{=0}      
\end{align*}
$$
Specifically, we need ${\epsilon\over \lambda} 2u v = u^2 + v^2$. The hint says to use something similar to class,

$$
    u = \cos \theta
\\  v = \sin \theta
\\  2 \cos \theta \sin \theta = \sin (2\theta)
\\  {\epsilon \over \lambda} \sin(2\theta) = 1
\\  2\theta = \sin^{-1}\left({\lambda \over \epsilon}\right)
$$
which is valid for $|\lambda| \leq |\epsilon|$. For $|\lambda| > |\epsilon|$, we have

$$
    u = \cosh \theta
\\  v = \sinh \theta
\\  {\epsilon \over \lambda} = {\cosh(2\theta) \over \sinh(2\theta)}
\\  2\theta = \tanh^{-1}\left( \epsilon \over \lambda \right)
$$

Rather than computing twice, let's find the fermi pseudoparticle energies in terms of $u$ and $v$.

$$
\begin{align*}
      \langle n_1, n_2 | \hat H | n_1, n_2 \rangle 
  & =(\epsilon u^2 + 2\lambda v) (n_1)
    + \epsilon v^2 (1-n_1)
\\& +(\epsilon v^2 + 2\lambda u) (1-n_2)
    + \epsilon u^2 (n_2)
\end{align*}
\\  = \left(\epsilon \left(u^2-v^2\right) + 2 \lambda v \right) n_1
    + \left(\epsilon \left(u^2-v^2\right) - 2 \lambda u \right)n_2
    + 2\epsilon v^2 + 2\lambda u
$$
yielding the vacuum energy $2 \epsilon v^2 + 2\lambda u$.

3. 
All transformations sounds like a lot. Logically, the two-state system does not allow for double creation or double annihilation, et cetera. I'll follow from the hint and calculate each coefficient. I'll assume the coefficients are real, and if that doesn't work we can go backwards.

In [26]:
a = Operator('a')
u, v, w, q = sp.symbols('u v w q', real=True)

b = u*a + v*Dagger(a) + w * Dagger(a) * a + q
anticommutator = b*Dagger(b) + Dagger(b)*b

display(anticommutator)
ac = collect_operators(anticommutator.expand(), collect_coeffs = True)
ac_collected = sum([(coeff * op) for op, coeff in ac.items() ])

display(ac_collected)

(q + u*Dagger(a) + v*a + w*Dagger(a)*a)*(q + u*a + v*Dagger(a) + w*Dagger(a)*a) + (q + u*a + v*Dagger(a) + w*Dagger(a)*a)*(q + u*Dagger(a) + v*a + w*Dagger(a)*a)

2*q**2 + 2*q*(u + v)*(Dagger(a) + a) + 2*u*v*(Dagger(a)**2 + a**2) + 2*w**2*Dagger(a)*a*Dagger(a)*a + w*(u + v)*(Dagger(a)*a*Dagger(a) + Dagger(a)*a**2 + Dagger(a)**2*a + a*Dagger(a)*a) + (u**2 + v**2)*a*Dagger(a) + (4*q*w + u**2 + v**2)*Dagger(a)*a

With the obvious zero terms removed, this becomes
$$
\def\ac#1#2{\left\{#1,#2\right\}}

\begin{align}
\ac{b}{b^\dag} &=
      2 q^{2}     
    + 2 q \left(u + v\right) \left(a^{\dagger} + a\right) 
\\ &+ 2 w^{2} a^{\dagger} a a^{\dagger} a 
    + w \left(u + v\right) \left(a^{\dagger} a a^{\dagger} + a a^{\dagger} a\right) 
\\ &+ \left(u^{2} + v^{2}\right) a a^{\dagger} 
    + \left(4 q w + u^{2} + v^{2}\right) a^{\dagger} a
\end{align}
$$
To proceed, we'll apply anticommutator relations to simplify line (2), then see what falls out.

$$
%    a^\dag a = 1 - a a^\dag
\\  \begin{align}
\ac{b}{b^\dag} &=
      2 q^{2}     
    + 2 q \left(u + v\right) \left(a^{\dagger}+ a\right) 
\\ &+ 2 w^{2} (1 - a a^\dag) a^{\dagger} a 
    + w \left(u + v\right) \left((1 - a a^\dag) a^{\dagger} + a (1 - a a^\dag)\right) 
\\ &+ \left(u^{2} + v^{2}\right) a a^{\dagger} 
    + 4 q w a^\dag a 
    + \left(u^{2} + v^{2}\right) (1- a a^\dag)
\end{align}
$$

$$
\\ \begin{align}
               &=
      2 q^{2}     
    + 2 q \left(u + v\right) \left(a^{\dagger}+ a\right) 
\\ &+ 2 w^{2} a^{\dagger} a 
    + w \left(u + v\right) \left( a^{\dagger} + a \right) 
\\ &
    + 4 q w a^\dag a 
    + \left(u^{2} + v^{2}\right) 
\end{align}
$$
Okay, this is looking more doable. Now let's see if any neat restrictions appear

$$
\ac{b}{b^\dag} 
    = 2q^2 
    + (u+v)(2q+w)\left(a^{\dagger}+ a\right) 
    + 2w   (2q+w)a^\dag a
    = 1
$$
To zero out the remaining operators, we can use $2q^2 = 1 \rightarrow q= \pm1/\sqrt{2}$, which binds $w=-2q=\mp2/\sqrt{2}$. Alternatively, we can use $w=0$ and $u=-v$

I missed a piece when doing this answer. Note that we should also observe

$$
b^2 = 0
$$

which gives

In [24]:
b2 = b*b
display(b2)
b2_collected = collect_operators(b2.expand(), collect_coeffs = True)
b2_v = sum([(coeff * op) for op, coeff in ac.items() ])

display(b2_v)

(q + u*a + v*Dagger(a) + w*Dagger(a)*a)**2

q**2 + 2*q*u*a + 2*q*v*Dagger(a) + u**2*a**2 + u*v*a*Dagger(a) + u*w*(Dagger(a)*a**2 + a*Dagger(a)*a) + v**2*Dagger(a)**2 + v*w*(Dagger(a)*a*Dagger(a) + Dagger(a)**2*a) + w**2*Dagger(a)*a*Dagger(a)*a + (2*q*w + u*v)*Dagger(a)*a

which yields
$$
\begin{align}
b^2&= q^{2} 
    + 2 q u a 
    + 2 q v a^{\dagger} 
\\ &+ u v a a^{\dagger} 
    + \left(2 q w + u v\right) a^{\dagger} a
\\ &+ u w \left(a a^{\dagger} a\right) 
    + v w \left(a^{\dagger} a a^{\dagger} \right) 
    + w^{2} \left(a^{\dagger} a\right)^{2} 
\end{align}
$$
$$
\begin{align}
   &= q^{2} 
    + 2 q u a 
    + 2 q v a^{\dagger} 
\\ &+ u v a a^{\dagger} 
    + 2 q w a^{\dagger} a 
    + u v \left(1- a a^{\dagger}\right)
\\ &+ u w \left(a \left(1-a a^{\dagger} \right)\right) 
    + v w \left(\left(1- a a^{\dagger} \right) a^{\dagger} \right) 
    + w^{2} \left(1- a a^{\dagger} \right) a^{\dagger} a
\end{align}
$$
$$
\begin{align}
   &= q^{2} 
    + 2 q u a 
    + 2 q v a^{\dagger} 
\\ & 
    + 2 q w a^{\dagger} a 
    + u v
\\ &+ u w a
    + v w a^\dagger
    + w^{2} a^\dag a
\end{align}
$$
$$  
b^2 
    = q^2
    + u v
    + (2 q + w) u a 
    + (2 q + w) v a^{\dagger} 
    + (2 q + w) w a^{\dagger} a 
\\  = q^2
    + u v
    + (2 q + w) \left(u a + v a^{\dagger} +  w a^{\dagger} a \right)
    = 0
$$

This means our $q=0$ solution is trivial, since $q^2 + uv =0$, would result in $u$ and $v$ being zero. As for the $q=\pm 1/\sqrt{2}$ this is valid and binds $u = -\frac{1}{2v}$, yielding

$$
b = {1 \over 2v} a - v a^\dag \mp {2 \over \sqrt 2} a^\dag a \pm {1 \over \sqrt{2}}
$$

4.

a) easy.

b) 
$$
\def\ac#1#2{\left\{#1,#2\right\}}
      b_k 
    = u_k a_k 
    + v_k a_{-k}^\dag
\\    b_{-k} 
    = u_{-k} a_{-k} 
    + v_{-k} a_k^\dag
$$
got it.

c) we want $\ac{b_i}{b_j^\dag} = \delta_{ij}$ and $b_i^2 = 0$

<!-- $$
      b_k^\dag 
    = u_k a_k^\dag 
    + v_k a_{-k}^\dag
\\    \ac{b_k}{b_k^\dag} 
    = \left( u_k a_k + v_k a_{-k}^\dag \right)
      \left( u_k a_k^\dag + v_k a_{-k}^\dag \right)
\\  = u_k^2 a_k a_k^\dag + v_k u_k a_{-k}^\dag  a_k^\dag
    + u_k v_k a_k a_{-k}^\dag + v_k^2 a_{-k}^\dag a_{-k}^\dag
$$ -->

We should use sympy since I already did the work to write all that code

In [23]:
uk, u_k, vk, v_k = sp.symbols('u_k u_{-k} v_k v_{-k}', real=True)
ak, a_k = Operator('a_k'), Operator('a_{-k}')

bk = uk*ak + vk*Dagger(a_k)
b_k = u_k*a_k + v_k*Dagger(ak)

pos_commutator = bk * Dagger(bk) + Dagger(bk) * bk
neg_commutator = b_k * Dagger(b_k) + Dagger(b_k) * b_k 

anticommuting_pairs = {
    frozenset([ak, a_k]),
    frozenset([ak, Dagger(a_k)]),
    frozenset([Dagger(ak), a_k]),
    frozenset([Dagger(ak), Dagger(a_k)]),
}

pos_collected = collect_operators(sp.expand(pos_commutator), collect_coeffs = True, anticommuting_pairs=anticommuting_pairs)
neg_collected = collect_operators(sp.expand(neg_commutator), collect_coeffs = True, anticommuting_pairs=anticommuting_pairs)
b_k_commutator  = sum([(coeff * op) for op, coeff in pos_collected.items() ])
b_negk_commutator  = sum([(coeff * op) for op, coeff in neg_collected.items() ])

print('{bk,bk†}:')
display(sp.Eq(b_k_commutator,1))
print('{b-k,b-k†}:')
display(sp.Eq(b_negk_commutator,1))


{bk,bk†}:


Eq(u_k**2*(Dagger(a_k)*a_k + a_k*Dagger(a_k)) + v_k**2*(Dagger(a_{-k})*a_{-k} + a_{-k}*Dagger(a_{-k})), 1)

{b-k,b-k†}:


Eq(u_{-k}**2*(Dagger(a_{-k})*a_{-k} + a_{-k}*Dagger(a_{-k})) + v_{-k}**2*(Dagger(a_k)*a_k + a_k*Dagger(a_k)), 1)

Easy-peasy! Anticommutation between lattice sites automatically took care of off-diagonal terms, and what's left is anticommuting fermions $\ac{a_i}{a_i^\dag}=1$

$$
    u_k^2 + v_k^2 = 1
\\  u_{-k}^2 + v_{-k}^2 = 1
$$

d) We should also check the fermion nilpotence $b_{i}^2 = 0$

In [21]:
b2k  = collect_operators(sp.expand(bk*bk), collect_coeffs = True, anticommuting_pairs=anticommuting_pairs)
b2_k = collect_operators(sp.expand(bk*bk), collect_coeffs = True, anticommuting_pairs=anticommuting_pairs)

b2k_expression  = sum([(coeff * op) for op, coeff in b2k.items() ])
b2_k_expression = sum([(coeff * op) for op, coeff in b2_k.items()])
        
display(sp.Eq(b2k_expression,0), sp.Eq(b2_k_expression,0))

Eq(u_k**2*a_k**2 + v_k**2*Dagger(a_{-k})**2, 0)

Eq(u_k**2*a_k**2 + v_k**2*Dagger(a_{-k})**2, 0)

This doesn't contribute any restrictions. To mix and create single-parameter dependence, we need to commutate $b_{k}$ and $b_{-k}$

$$
\ac{b_k}{b_{-k}} = 0
$$

In [27]:
mixing_commutator = bk * b_k + b_k * bk
mixing_collected  = collect_operators(sp.expand(mixing_commutator), collect_coeffs = True, anticommuting_pairs=anticommuting_pairs)
m = sum([(coeff * op) for op, coeff in mixing_collected.items()])

display(sp.Eq(m,0))

Eq(u_k*v_{-k}*(Dagger(a_k)*a_k + a_k*Dagger(a_k)) + u_{-k}*v_k*(Dagger(a_{-k})*a_{-k} + a_{-k}*Dagger(a_{-k})), 0)

So the mixing requirement is $u_k v_{-k} + u_{-k}v_k = 0$. Separately, we have

$$
    u = \cos \theta
\\  v = \sin \theta
$$
so, by the powers of trigonometry, we have

$$
\cos(\vartheta_k)\sin(\theta_{-k}) + \cos(\theta_{-k})\sin(\vartheta_k) = \sin(\vartheta_k + \theta_{-k}) = 0
\\  \vartheta_k + \theta_{-k} = \arcsin(0) = 0
\\  \theta_{-k} = -\vartheta_{k}
$$
yielding the parameterization
$$
    b_k = \cos (\vartheta_k) a_k + \sin (\vartheta_k) a_{-k}^\dag
\\  b_{-k} = \cos(-\vartheta_k) a_{-k} + \sin(-\vartheta_k) a_k^\dag
\\         = \cos(\vartheta_k) a_{-k} - \sin(\vartheta_k) a_k^\dag
$$

e) 
$$
a_k = \frac{b_k - \sin(\vartheta_k)a^\dag_{-k}}{\cos(\vartheta_k)}
$$
Now, to get rid of $a_{-k}^\dag$
$$
    % b_k^\dag = \cos(\vartheta_k)a_k^\dag + \sin(\vartheta_k)a_{-k}
\\  b_{-k}^\dag = \cos(\vartheta_k)a_{-k}^\dag - \sin(\vartheta_k) a_k
\\  \frac{b_{-k}^\dag + \sin(\vartheta_k) a_k}{\cos(\vartheta_k)}=a_{-k}^\dag 
\\  
a_k 
    = \frac{b_k}{\cos(\vartheta_k)} 
    - \frac{\sin(\vartheta_k)\frac{b_{-k}^\dag + \sin(\vartheta_k) a_k}{\cos(\vartheta_k)}}{\cos(\vartheta_k)}

$$

5. 
(a) show commutation & anticommutation relations. 

$$
\def\c#1#2{\left[#1,#2\right]}
\def\ac#1#2{\left\{#1,#2\right\}}

      \c{Q}{H} 
    = QH - HQ
    = \omega b^\dag a a^\dag a 
    + \omega b^\dag a b^\dag b
    - \omega a^\dag a b^\dag a
    - \omega b^\dag b b^\dag a
\\  = \omega \left(
          b^\dag a a^\dag a
        - b^\dag a^\dag a a
        + b^\dag b^\dag b a 
        - b^\dag b b^\dag a
    \right)

\\  = \omega \left(
          b^\dag \c{a}{a^\dag} a
        + b^\dag b^\dag b a 
        - b^\dag b b^\dag a
    \right)
$$
The boson/fermion commutators also say
$$
    a a^\dag = 1 + a^\dag a
\\  b b^\dag = 1 - b^\dag b
$$
yielding
$$
\\    \c{Q}{H} 
    = \omega \left(
          b^\dag \c{a}{a^\dag} a
        + b^\dag b^\dag b a 
        - b^\dag (1- b^\dag b) a
    \right)
\\  = \omega \left(
          b^\dag a
        + b^\dag b^\dag b a 
        - b^\dag  a
        + b^\dag b^\dag b a
    \right)
\\  = 2 \omega b^\dag b^\dag b a
$$
I think that $b^\dag b^\dag = bb = 0 $ since fermions can only be in the $n_F = 0$ or $1$ states
$$
\c{Q}{H} = 0
$$

<!-- Fermionic operators commute, yielding $b^\dag b^\dag b a - b^\dag b^\dag b a$, so the remaining terms are

$$
    = \omega \left(
          b^\dag a a^\dag a
        - b^\dag a^\dag a a \right)
\\  = \omega b^\dag \left( \c{a}{a^\dag} \right)a
\\  = \omega b^\dag a = \omega Q
$$
wtf -->

For $\c{Q^\dag}{H}$, let's speed along
$$
      \c{Q^\dag}{H} 
    = \omega \left(
      ba^\dag a^\dag a 
    + ba^\dag b^\dag b 
    - a^\dag a a^\dag b 
    - b^\dag b a^\dag b
    \right)
\\  = \omega \left(
      b a^\dag a^\dag a 
    - b a^\dag a a^\dag 
    + b b^\dag b a^\dag
    - b^\dag b b a^\dag
    \right)
\\  = \omega \left(
    - Q^\dag
    + (1-b^\dag b) Q^\dag
    - 0
    \right)
\\  = \omega(Q^\dag - Q^\dag - 0) = 0
$$

Next, we have
$$
      \ac{Q}{Q^\dag}
    = b^\dag a b a^\dag + b a^\dag b^\dag a    
    = b^\dag b a a^\dag + b b^\dag a^\dag  a    
\\  = b^\dag b (1 + a^\dag a) + (1 - b^\dag b) a^\dag  a    
\\  = b^\dag b  + b^\dag b a^\dag a + a^\dag  a - b^\dag b a^\dag  a    
\\  = b^\dag b  + a^\dag  a    
    = {1 \over \omega} H
\\  \beta = {1 \over \omega}
$$


Using our previous results, we have



$$
      \ac{Q_1 + iQ_2}{Q_1-iQ_2} 
    = {1 \over \omega} H
$$
Expanding this out,
$$
      \ac{Q_1 + iQ_2}{Q_1-iQ_2}
    = \ac{Q_1}{Q_1-iQ_2} + \ac{iQ_2}{Q_1-iQ_2}
\\  = \ac{Q_1}{Q_1} -\ac{Q_1}{iQ_2} + \ac{iQ_2}{Q_1} - \ac{iQ_2}{iQ_2}
\\  = \ac{Q_1}{Q_1} -\ac{Q_1}{iQ_2} + \ac{Q_1}{iQ_2} - i^2\ac{Q_2}{Q_2}
\\  = \ac{Q_1}{Q_1} + \ac{Q_2}{Q_2} 
    = {1 \over \omega} H
$$
This result isn't quite complete. Let's try something else
$$
    Q_1 = {Q+Q^\dag \over 2}
\\  Q_2 = {Q-Q^\dag \over 2i}
\\  QH-HQ = (Q_1 + iQ_2)H - H(Q_1 + iQ_2)
\\  
$$

(b) This is obvious? Maybe write in terms of the Fock basis?

$$
\def\ket#1{|#1\rangle}
    \ket{1,0} = a^\dag \ket{0,0}
\\  \ket{0,1} = b^\dag \ket{0,0}
\\  Q\ket{1,0} = b^\dag a a^\dag \ket{0,0}
               = 2 b^\dag \ket{0,0}
               = 2 \ket{0,1}
$$

(c) 

To speed things along, we'll use the anticommutator identity
$$
\def\ac#1#2{\left\{#1,#2\right\}}

\ac{a+b}{a-b} = 2\left(a^2-b^2\right)
$$
yielding
$$
    b^\dag = {1 \over 2} \left( \sigma_x + i \sigma_y \right)
\\    \ac{b}{b^\dag} 
    = {1 \over 2} \left( \sigma_x^2 - i^2 \sigma_y^2 \right) 
    = {1 \over 2} \left( 1 + 1 \right) = 1 
$$
This gives
$$
b^\dag b = {1 \over 4} \left(\sigma_x^2 + i\sigma_y\sigma_x + i\sigma_x \sigma_y + \sigma_y^2\right)
\\       = {1 \over 4} \left(\sigma_x^2 + \sigma_y^2 + i\ac{\sigma_y}{\sigma_x} \right)
         = {1 \over 2}
$$

The Hamiltonian is then

$$
    a^\dag a = {1 \over \hbar \omega} \left({p^2 \over 2m} + {1 \over 2} m\omega^2 x^2 \right) -{1\over 2}
\\  H = {1 \over \hbar \omega} \left({p^2 \over 2m} + {1 \over 2} m\omega^2 x^2 \right) 
$$